In [39]:
from rdflib import Graph, Namespace, RDF, OWL, XSD, Literal
import pandas as pd
g = Graph()

# Load existing graph
g.parse('../graphs/final_kg.ttl', format='turtle')

print(f"Loaded {len(g)} triples from existing graph")



Loaded 7263 triples from existing graph


In [35]:
from rdflib.plugins.sparql import prepareQuery

queries = """
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dance: <http://example.org/dance/>
SELECT ?danceStyle ?hardness
WHERE {
 ?record a dance:DanceRecord ;
        dance:hasDanceStyle ?danceStyle ;
        dance:hardnessRatio ?hardness .
 FILTER(xsd:decimal(?hardness) <0.5)
}
#ORDER BY DESC(?hardness)
"""

query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

,danceStyle,hardness
0,http://example.org/dance/Khaleeji,0.4
1,http://example.org/dance/Kagura,0.3
2,http://example.org/dance/Medieval_dance,0.3
3,http://example.org/dance/Regency_dance,0.4
4,http://example.org/dance/Bachata,0.4
5,http://example.org/dance/Cumbia,0.3
6,http://example.org/dance/Merengue,0.4
7,http://example.org/dance/Bunny_hop,0.2
8,http://example.org/dance/Conga_line,0.3
9,http://example.org/dance/Madison,0.4


In [11]:
from rdflib.plugins.sparql import prepareQuery

queries = """
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX dance: <http://example.org/dance/>
PREFIX schema1: <http://schema.org/>

SELECT ?danceName ?instrumentName
WHERE {

 ?record a dance:DanceRecord ;
        dance:hasDanceStyle ?danceStyle ;
        dance:hasInstrument ?instrument .
 ?danceStyle schema1:name ?danceName.
 ?instrument schema1:name ?instrumentName
 FILTER(?instrument = dance:Drums)
}
#ORDER BY DESC(?hardness)
"""

query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)


""


In [44]:
from rdflib.plugins.sparql import prepareQuery

# Count total triples, distinct dance styles, instruments, music genres, origins, practitioners
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT
  (COUNT(DISTINCT ?style)            AS ?distinctDanceStyles)
  (COUNT(DISTINCT ?instr)            AS ?distinctInstruments)
  (COUNT(DISTINCT ?genre)            AS ?distinctMusicGenres)
  (COUNT(DISTINCT ?origin)           AS ?distinctOrigins)
  (COUNT(DISTINCT ?pract)            AS ?distinctPractitioners)
WHERE {
  { SELECT * WHERE { ?s ?p ?o } }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasDanceStyle           ?style  . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasInstrument           ?instr  . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasAssociatedMusicGenre ?genre  . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasOrigin               ?origin . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasFamousPractitioner   ?pract  . }
}
"""

query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

,distinctDanceStyles,distinctInstruments,distinctMusicGenres,distinctOrigins,distinctPractitioners
0,127,110,105,36,130


In [ ]:
#here

In [43]:
from rdflib.plugins.sparql import prepareQuery

# Count distinct values for all multi-instance object properties
queries = """
PREFIX dance: <http://example.org/dance/>

SELECT
  (COUNT(DISTINCT ?hb)    AS ?distinctHealthBenefits)
  (COUNT(DISTINCT ?instr) AS ?distinctInstruments)
  (COUNT(DISTINCT ?genre) AS ?distinctMusicGenres)
  (COUNT(DISTINCT ?pract) AS ?distinctPractitioners)
  (COUNT(DISTINCT ?event) AS ?distinctEventFestivals)
  (COUNT(DISTINCT ?tp)    AS ?distinctTimePeriods)
  (COUNT(DISTINCT ?rec)    AS ?rec)
  (COUNT(DISTINCT ?style)    AS ?distinctStyle)


WHERE {
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasHealthBenefit         ?hb    . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasInstrument            ?instr . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasAssociatedMusicGenre  ?genre . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasFamousPractitioner    ?pract . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasEventFestival         ?event . }
  OPTIONAL { ?rec a dance:DanceRecord ; dance:hasTimePeriod            ?tp    . }
OPTIONAL { ?rec a dance:DanceRecord ; dance:hasDanceStyle ?style . ?style dance:hasYTLink ?yt . }

}
"""

query  = prepareQuery(queries)
result = g.query(query)
pd.DataFrame(result.bindings)

,distinctEventFestivals,distinctHealthBenefits,distinctInstruments,distinctMusicGenres,distinctPractitioners,distinctStyle,distinctTimePeriods,rec
0,190,94,110,105,130,124,29,130


In [31]:
from rdflib.plugins.sparql import prepareQuery

# Full details for a single dance record
queries = """
PREFIX dance:   <http://example.org/dance/>
PREFIX schema1: <http://schema.org/>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>
PREFIX wd:      <http://www.wikidata.org/entity/>

SELECT ?styleName ?typeName ?originName ?difficulty ?minBPM ?maxBPM
       ?formation ?ageGroup ?healthBenefit ?instrument ?genre
       ?practitioner ?event ?timePeriod ?costume ?culturalSignificance
       ?notableCharacteristics ?modernAdaptations ?hardnessRatio
       ?videoTitle ?viewCount ?sentiment
WHERE {
  ?rec a dance:DanceRecord ;
       dance:hasDanceStyle         ?style ;
       dance:hasDanceType          ?type ;
       dance:hasLearningDifficulty ?diff ;
       dance:hasTempoRange         ?tempo ;
       dance:hasDanceFormation     ?form ;
       dance:hasAgeGroup           ?age .

  ?style schema1:name ?styleName .
  ?type  schema1:name ?typeName .
  ?diff  schema1:name ?difficulty .
  ?form  schema1:name ?formation .
  ?age   schema1:name ?ageGroup .
  ?tempo dance:minTempoBPM ?minBPM ;
         dance:maxTempoBPM ?maxBPM .

  OPTIONAL { ?rec dance:hasOrigin               ?orig  . ?orig  schema1:name ?originName . }
  OPTIONAL { ?rec dance:hasHealthBenefit         ?hb    . ?hb    schema1:name ?healthBenefit . }
  OPTIONAL { ?rec dance:hasInstrument            ?instr . ?instr schema1:name ?instrument . }
  OPTIONAL { ?rec dance:hasAssociatedMusicGenre  ?mg    . ?mg    schema1:name ?genre . }
  OPTIONAL { ?rec dance:hasFamousPractitioner    ?pr    . ?pr    schema1:name ?practitioner . }
  OPTIONAL { ?rec dance:hasEventFestival         ?ev    . ?ev    schema1:name ?event . }
  OPTIONAL { ?rec dance:hasTimePeriod            ?tp    . ?tp    schema1:name ?timePeriod . }
  OPTIONAL { ?rec dance:costume                  ?costume . }
  OPTIONAL { ?rec dance:culturalSignificance     ?culturalSignificance . }
  OPTIONAL { ?rec dance:notableCharacteristics   ?notableCharacteristics . }
  OPTIONAL { ?rec dance:modernAdaptations        ?modernAdaptations . }
  OPTIONAL { ?rec dance:hardnessRatio            ?hardnessRatio . }
  OPTIONAL {
    ?style dance:hasYTLink ?yt .
    ?yt schema1:title            ?videoTitle .
    ?yt schema1:interactionCount ?viewCount .
    ?yt wd:Q28659447             ?sentiment .
  }
}
LIMIT 1
"""

query  = prepareQuery(queries)
result = g.query(query)
pd.set_option('display.max_columns', None)
df = pd.DataFrame(result.bindings)
df.to_csv('../graphs/single_dance_record.csv', index=False)
df.T

,0
ageGroup,All ages
costume,"Flexible, form-fitting attire to allow freedom..."
culturalSignificance,Jazz dance evolved alongside jazz music as a f...
difficulty,Intermediate
event,Jazz festivals
formation,Solo
genre,Blues
hardnessRatio,0.5
healthBenefit,Cardiovascular health
instrument,Body percussion


In [38]:
from rdflib.plugins.sparql import prepareQuery

# Full details for a single dance record
queries = """
PREFIX dance:   <http://example.org/dance/>
PREFIX schema1: <http://schema.org/>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>
PREFIX wd:      <http://www.wikidata.org/entity/>

SELECT ?videoTitle ?styleName ?yt ?sentiment ?uploadDate ?UserLikes ?description ?duration
WHERE {
  ?rec a dance:DanceRecord ;
       dance:hasDanceStyle         ?style ;
       dance:hasDanceType          ?type ;
       dance:hasLearningDifficulty ?diff ;
       dance:hasTempoRange         ?tempo ;
       dance:hasDanceFormation     ?form ;
       dance:hasAgeGroup           ?age .

  ?style schema1:name ?styleName .

  OPTIONAL {
    ?style dance:hasYTLink ?yt .

    ?yt schema1:title            ?videoTitle .
    ?yt schema1:interactionCount ?viewCount .
    ?yt wd:Q28659447             ?sentiment .
    ?yt schema1:UserLikes ?likes .
    ?yt schema1:description ?description .
    ?yt schema1:uploadDate ?uploadDate .
    ?yt xsd:duration ?duration .
  }
}
LIMIT 1
"""

query  = prepareQuery(queries)
result = g.query(query)
pd.set_option('display.max_columns', None)
df = pd.DataFrame(result.bindings)
df.to_csv('../graphs/single_dance_record.csv', index=False)
df.T

,0
description,Follow me and let's move together for a bit. T...
duration,PT4M9S
sentiment,positive
styleName,Jazz dance
uploadDate,2022-02-03
videoTitle,Solo Jazz warm-up dance to Honeysuckle Rose
yt,https://www.youtube.com/watch?v=B-8kxNao4aw
